# Stage 1 zero-shot evaluation (Colab)

Runs the AVOS-trained YOLOv8 instrument detector zero-shot on the hypospadias eval
set (and the AVOS test split, for the accuracy comparison), and produces the
per-class / pooled / pairwise-comparison report.

**Before running:** Runtime menu -> Change runtime type -> GPU (not required, but much faster).

## 1. Clone the repo

In [ ]:
!git clone https://github.com/cxia0024/hypospadias-object-detection.git
%cd hypospadias-object-detection
!git checkout claude/surgical-phase-recognition-wqrp7r

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Sanity check: run the unit tests

These only exercise the statistics module -- no data or model needed -- so this
should pass before you go any further.

In [ ]:
!python -m pytest tests/ -q

## 4. Mount Google Drive

Put your YOLOv8 checkpoint, frame images, and label CSVs somewhere in Drive
(e.g. `MyDrive/hypospadias/data/...`, `MyDrive/hypospadias/models/...`) and mount it here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4b. (If needed) Extract frames from raw videos

Skip this if you already have `images_dir` frame folders and a filled-in
`labels_csv` for each dataset. Otherwise, this randomly samples frames from
your source videos and writes an **unlabeled** template -- extraction does not
produce ground truth, an expert still has to fill in presence/absence.

In [ ]:
import sys
sys.path.insert(0, "src")
from stage1_detection.extract_frames import extract_random_frames, write_manifest, write_labeling_template
from stage1_detection.predict import get_model_classes

VIDEOS_DIR = f"{DRIVE_ROOT}/videos/hypospadias"      # <-- change to your raw video folder
FRAMES_OUT = f"{DRIVE_ROOT}/data/hypospadias_eval/images"
MODEL_PATH = f"{DRIVE_ROOT}/models/yolov8_avos_best.pt"

# Classes come from the checkpoint itself (the AVOS bounding-box classes it was
# trained on) -- never hardcoded here.
CLASSES = get_model_classes(MODEL_PATH)
print("Classes from checkpoint:", CLASSES)

rows = extract_random_frames(VIDEOS_DIR, FRAMES_OUT, n_per_video=8, seed=42)
write_manifest(rows, f"{DRIVE_ROOT}/data/hypospadias_eval/manifest.csv")
write_labeling_template(rows, CLASSES, f"{DRIVE_ROOT}/data/hypospadias_eval/labels_template.csv")

print("\nNEXT STEP: have an expert fill in the 'label' column (0/1) in "
      "labels_template.csv, save the completed file as labels.csv, then "
      "point hypospadias_eval.labels_csv at it in the config cell below. "
      "The eval pipeline will refuse to run on the unfilled template.")

## 5. Point the config at your data

Edit the paths below to match where your files live in Drive, then this cell
rewrites `configs/stage1_datasets.yaml` with those paths. Expected layout per dataset:
- `images_dir`: a folder of frame images
- `labels_csv`: expert presence/absence labels with columns `frame_id,class,label`

`model_path` should point at your AVOS-trained YOLOv8 checkpoint (`.pt` file).

In [ ]:
import yaml

DRIVE_ROOT = "/content/drive/MyDrive/hypospadias"  # <-- change to your Drive folder

config = {
    "model_path": f"{DRIVE_ROOT}/models/yolov8_avos_best.pt",
    "conf_threshold": 0.25,
    # classes are read from the checkpoint's own names at eval time -- no need to
    # hardcode them here. Only add a "classes": [...] key if you want to score a
    # subset of what the model was trained on.
    "datasets": {
        "avos_test": {
            "images_dir": f"{DRIVE_ROOT}/data/avos/images/val",
            "labels_csv": f"{DRIVE_ROOT}/data/avos/avos_test_labels.csv",  # from validate_avos_colab.ipynb
            "chance_level": 0.5,
        },
        "hypospadias_eval": {
            "images_dir": f"{DRIVE_ROOT}/data/hypospadias_eval/images",
            "labels_csv": f"{DRIVE_ROOT}/data/hypospadias_eval/labels.csv",
            "chance_level": 0.5,
        },
    },
}

with open("configs/stage1_datasets.yaml", "w") as f:
    yaml.dump(config, f, sort_keys=False)

print(open("configs/stage1_datasets.yaml").read())

## 6. Run the evaluation

In [ ]:
# Assumes avos_test/labels.csv came from validate_avos_colab.ipynb and
# hypospadias_eval/labels.csv has been expert-labeled (see step 4b above).
!python scripts/run_stage1_eval.py --config configs/stage1_datasets.yaml --out results/stage1

## 7. Inspect the results

In [ ]:
import pandas as pd

pooled = pd.read_csv("results/stage1/pooled_metrics.csv")
per_class = pd.read_csv("results/stage1/per_class_metrics.csv")
pairwise = pd.read_csv("results/stage1/pairwise_comparisons.csv")

print("Pooled accuracy per dataset:")
display(pooled)

print("\nPer-class metrics:")
display(per_class)

print("\nAVOS vs. hypospadias (and any other configured pair), two-proportion z-test:")
display(pairwise)

## 8. Save results back to Drive (optional)

In [ ]:
!cp -r results/stage1 "$DRIVE_ROOT/results_stage1"
print(f"Copied to {DRIVE_ROOT}/results_stage1")